In [1]:
from pathlib import Path

import duckdb
import pandas as pd

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'bin').is_dir():
    PROJECT_DIR = PROJECT_DIR.parent
DATABASE_PATH = PROJECT_DIR / 'local_data' / 'ani_microbial_eukaryotes.duckdb'
if not DATABASE_PATH.is_file():
    raise FileNotFoundError(DATABASE_PATH)

EXPECTED_QC_PASS_GENOMES = 7_665
TOP_PAIRS_PER_CATEGORY = 10
CATEGORY_ORDER = [
    'Intraspecies, ANI <95',
    'Interspecies, ANI >95',
    'Interspecies, ANI 85–95',
]


In [2]:
con = duckdb.connect(str(DATABASE_PATH), read_only=True)
try:
    con.execute('SET max_expression_depth=100000')
    qc_pass_genomes = con.execute(
        "SELECT count(*) FROM genomes",
    ).fetchone()[0]
    if qc_pass_genomes != EXPECTED_QC_PASS_GENOMES:
        raise RuntimeError(
            f'Expected {EXPECTED_QC_PASS_GENOMES:,} QC-pass genomes; found {qc_pass_genomes:,}'
        )
    print(f'QC-pass genomes: {qc_pass_genomes:,}')
    outlier_table = con.execute(
        f"""
        WITH eligible_pairs AS (
            SELECT
                p.ani AS ANI,
                m1.species_2026_09_01 AS species1,
                m2.species_2026_09_01 AS species2
            FROM pairwise_metrics p
            JOIN genomes m1 ON p.genome1_id = m1.genome_id
            JOIN genomes m2 ON p.genome2_id = m2.genome_id
            WHERE p.ani IS NOT NULL
              AND m1.species_2026_09_01 IS NOT NULL
              AND trim(m1.species_2026_09_01) <> ''
              AND m2.species_2026_09_01 IS NOT NULL
              AND trim(m2.species_2026_09_01) <> ''
        ),
        categorized AS (
            SELECT
                ANI,
                least(species1, species2) AS species1,
                greatest(species1, species2) AS species2,
                CASE
                    WHEN species1 = species2 AND ANI < 95
                        THEN 'Intraspecies, ANI <95'
                    WHEN species1 <> species2 AND ANI > 95
                        THEN 'Interspecies, ANI >95'
                    WHEN species1 <> species2 AND ANI BETWEEN 85 AND 95
                        THEN 'Interspecies, ANI 85–95'
                END AS outlier_category
            FROM eligible_pairs
        ),
        summarized AS (
            SELECT
                outlier_category,
                species1,
                species2,
                avg(ANI) AS mean_ani,
                count(*)::BIGINT AS comparison_count
            FROM categorized
            WHERE outlier_category IS NOT NULL
            GROUP BY outlier_category, species1, species2
        ),
        ranked AS (
            SELECT
                *,
                100.0 * comparison_count
                    / sum(comparison_count) OVER (PARTITION BY outlier_category)
                    AS category_share,
                row_number() OVER (
                    PARTITION BY outlier_category
                    ORDER BY comparison_count DESC, species1, species2
                ) AS category_rank
            FROM summarized
        )
        SELECT
            outlier_category AS "Outlier Category",
            CASE
                WHEN species1 = species2 THEN species1
                WHEN split_part(species1, ' ', 1) = split_part(species2, ' ', 1)
                    THEN species1 || ' / ' || substr(
                        species2, length(split_part(species2, ' ', 1)) + 2
                    )
                ELSE species1 || ' / ' || species2
            END AS "Species pair",
            mean_ani AS "Mean ANI",
            comparison_count AS "Comparisons (count)",
            category_share AS "Category Share (%)",
            CASE outlier_category
                WHEN 'Intraspecies, ANI <95' THEN 1
                WHEN 'Interspecies, ANI >95' THEN 2
                WHEN 'Interspecies, ANI 85–95' THEN 3
            END AS category_order,
            category_rank
        FROM ranked
        WHERE category_rank <= {TOP_PAIRS_PER_CATEGORY}
        ORDER BY category_order, category_rank
        """
    ).fetchdf()
finally:
    con.close()

outlier_table = outlier_table.drop(columns=['category_order', 'category_rank'])
outlier_table['Outlier Category'] = pd.Categorical(
    outlier_table['Outlier Category'], categories=CATEGORY_ORDER, ordered=True
)


QC-pass genomes: 7,665


In [3]:
display_table = outlier_table.copy()
display_table['Outlier Category'] = display_table['Outlier Category'].astype(str)
display_table.loc[display_table['Outlier Category'].duplicated(), 'Outlier Category'] = ''
(
    display_table.style
    .hide(axis='index')
    .format({
        'Mean ANI': '{:.3f}',
        'Comparisons (count)': '{:,.0f}',
        'Category Share (%)': '{:.2f}',
    })
    .set_properties(subset=['Species pair'], **{'font-style': 'italic'})
)


Outlier Category,Species pair,Mean ANI,Comparisons (count),Category Share (%)
"Intraspecies, ANI <95",Saccharomyces cerevisiae,93.081,"3,407",30.00
,Fusarium oxysporum,89.918,"2,983",26.27
,Aspergillus niger,89.106,"2,585",22.76
,Giardia duodenalis,78.107,307,2.70
,Aspergillus flavus,93.939,255,2.25
,Saccharomyces kudriavzevii,93.308,199,1.75
,Nakaseomyces glabratus,94.326,195,1.72
,Komagataella pastoris,90.137,192,1.69
,Fusarium equiseti,93.474,174,1.53
,Saccharomyces uvarum,92.948,95,0.84
